#GOLD LAYER: BI READY DATA

In [0]:
from pyspark.sql.types import StringType, IntegerType, DateType, BooleanType
import pyspark.sql.functions as F
from delta.tables import DeltaTable

dbutils.widgets.text("catalog_name", "ecommerce", "Catalog Name")
dbutils.widgets.text("storage_account_name", "storageaccount", "Storage Account Name")
dbutils.widgets.text("container_name", "ecomm-raw-data", "Container Name")

catalog_name = dbutils.widgets.get("catalog_name")
storage_account_name = dbutils.widgets.get("storage_account_name")
container_name = dbutils.widgets.get("container_name")

# print(catalog_name, storage_account_name, container_name)

ecommerce stgecommjackyind ecomm-raw-data


In [0]:
# Read incremental changes from Silver

df = (
    spark.readStream
    .format("delta")
    .option("readChangeFeed", "true")
    .table(f"{catalog_name}.silver.slv_order_items")
)

In [0]:
# Keep only active records
df_gold = df.filter(
    "_change_type IN ('insert', 'update_postimage')"
)

# Sales calculations
df_gold = (
    df_gold

    # Quantity × Unit Price
    .withColumn(
        "gross_amount",
        F.col("quantity") * F.col("unit_price")
    )

    # Discount amount
    .withColumn(
        "discount_amount",
        F.ceil(
            F.col("gross_amount")
            * (F.col("discount_pct") / 100.0)
        )
    )

    # Final sales amount
    .withColumn(
        "sale_amount",
        F.col("gross_amount")
        - F.col("discount_amount")
        + F.col("tax_amount")
    )

    # Date dimension key
    .withColumn(
        "date_id",
        F.date_format(
            F.col("dt"),
            "yyyyMMdd"
        ).cast("int")
    )

    # Coupon indicator
    .withColumn(
        "coupon_flag",
        F.when(
            F.col("coupon_code").isNotNull(),
            1
        ).otherwise(0)
    )
)

In [0]:
orders_gold_df = df_gold.select(
    F.col("date_id"),
    F.col("dt").alias("transaction_date"),
    F.col("order_ts").alias("transaction_ts"),
    F.col("order_id").alias("transaction_id"),
    F.col("customer_id"),
    F.col("item_seq").alias("seq_no"),
    F.col("product_id"),
    F.col("channel"),
    F.col("coupon_code"),
    F.col("coupon_flag"),
    F.col("unit_price_currency"),
    F.col("quantity"),
    F.col("unit_price"),
    F.col("gross_amount"),
    F.col("discount_pct").alias("discount_percent"),
    F.col("discount_amount"),
    F.col("tax_amount"),
    F.col("sale_amount").alias("net_amount")
)

In [0]:
gold_checkpoint_path = (
    f"abfss://{container_name}@{storage_account_name}"
    f".dfs.core.windows.net/checkpoint/gold/fact_order_items/"
)

def upsert_to_gold(microBatchDF, batchId):

    table_name = f"{catalog_name}.gold.gld_fact_order_items"

    # First run
    if not spark.catalog.tableExists(table_name):

        print("Creating Gold table...")

        microBatchDF.write \
            .format("delta") \
            .mode("overwrite") \
            .saveAsTable(table_name)

        spark.sql(
            f"""
            ALTER TABLE {table_name}
            SET TBLPROPERTIES
            (delta.enableChangeDataFeed = true)
            """
        )

    # Incremental merge
    else:

        deltaTable = DeltaTable.forName(
            spark,
            table_name
        )

        (
            deltaTable.alias("gold_table")
            .merge(
                microBatchDF.alias("batch_table"),
                """
                gold_table.transaction_id =
                batch_table.transaction_id

                AND

                gold_table.seq_no =
                batch_table.seq_no
                """
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )

In [0]:
# Process Silver changes and load Gold

(
    orders_gold_df.writeStream
    .foreachBatch(upsert_to_gold)
    .option(
        "checkpointLocation",
        gold_checkpoint_path
    )
    .outputMode("update")
    .trigger(availableNow=True)
    .start()
    .awaitTermination()
)

# # Validation

# spark.sql(
#     f"""
#     SELECT COUNT(*)
#     FROM {catalog_name}.gold.gld_fact_order_items
#     """
# ).show()

# spark.sql(
#     f"""
#     SELECT MAX(transaction_date)
#     FROM {catalog_name}.gold.gld_fact_order_items
#     """
# ).show()

+--------+
|COUNT(*)|
+--------+
| 1188115|
+--------+

+---------------------+
|MAX(transaction_date)|
+---------------------+
|           2025-08-31|
+---------------------+

